# Introduction

cuDF is a Python GPU DataFrame library (built on the Apache Arrow columnar memory format) for loading, joining, aggregating, filtering, and otherwise manipulating tabular data using a DataFrame style API in the style of pandas.

cuDF includes a pandas accelerator mode (`cudf.pandas`), enabling you to accelerate your pandas workflows without requiring any code change.

This notebook highlights the impact of GPU-acceleration for common operations and analytical questions analyzing a real-world dataset of stock prices.

For a deeper introduction and insight into how things work under the hood, we encourage you to run the [10 Minutes to RAPIDS cuDF's pandas accelerator mode Colab notebook](https://nvda.ws/rapids-cudf) or visit https://rapids.ai/cudf-pandas/.

# ⚠️ Verify your setup

First, we'll verify that you are running with an NVIDIA GPU.

In [ ]:
!nvidia-smi  # this should display information about available GPUs

With our GPU-enabled Colab runtime active, we're ready to go. cuDF is available by default in the GPU-enabled runtime.

# Download the data

The data we'll be working with is a subset of the [USA 514 Stocks Prices NASDAQ NYSE dataset](https://www.kaggle.com/datasets/olegshpagin/usa-stocks-prices-ohlcv) from Kaggle.

We'll start by downloading the dataset from NVIDIA's Public Google Cloud Storage bucket to provide faster download speeds to Colab.  This should take under 30 seconds.

In [ ]:
!if [ ! -f "usa_stocks_30m.parquet" ]; then curl https://storage.googleapis.com/rapidsai/colab-data/usa_stocks_30m.parquet -o usa_stocks_30m.parquet; else echo "usa_stocks_30m.parquet found"; fi

# Analysis using Standard Pandas



> **Microsoft Fabric note:** `%load_ext cudf.pandas` must be the very first executed statement — before any `import pandas`. In Fabric, pandas may be pre-imported by the environment. If so, add `cudf.pandas` to the Fabric session pre-run script or restart the kernel and ensure this cell runs first.

In [ ]:
%load_ext cudf.pandas

In [ ]:
import pandas as pd

In [ ]:
%time df = pd.read_parquet("usa_stocks_30m.parquet")
df.info()

In [ ]:
df.head()

We've got about 36M rows and 7 columns, covering stock price and trading-related attributes (`open`, `close`, etc.) for various companies (`ticker`) at what looks like 30 minute intervals.

Let's look at the more detailed summary statistics for the the numeric data.

To fit this analysis on Colab's free tier without running out of the 12GB of CPU memory, we'll only use the first 18 million rows of this dataset. If you'd like to process the full dataset, you can [sign up a Colab Pro account](https://colab.research.google.com/signup)!


In [ ]:
df = df.iloc[:18000000]

In [ ]:
%time df.describe()

A little slow, but it gets the job done.

Most of the price-related columns look well behaved with no crazy outliers based on the max values. Volume has a high variance, but that makes sense as some stocks will be traded **much** more often than others. We've also got a wide time range in this data, spanning 1998 to 2024.

The time-range information feels important. This is per-stock data, so some stocks will have "entered" the dataset at different times. We should really group by each ticker for our analysis.

To start, we can investigate the time periods for various stocks in the data.

In [ ]:
%time df.groupby("ticker").agg({"datetime": ["min", "max", "count"]})

As expected, there's a pretty significant difference across stocks. Since this determines how many records exist for each ticker, we'll need to take that into account for future analysis.

One way to do that is to use the time periods as part of any grouping or rolling window analysis we do on this data. For example, we can look at the minimum and maximum of each stock ticker at various time frequencies to understand trends.

In [ ]:
%%time
df[["year", "week", "day"]] = df.datetime.dt.isocalendar()
df.groupby(["ticker", "year", "week"]).agg({"close": ["min", "max"]})

Okay, that was a little slow, but it's manageable. We can easily aggregate the data up to weekly values and investigate further.

But what if we want to ask slightly more complex questions?

## More complex analysis


Let's say we want the daily rolling average of all of these values for each stock. With this, we could investigate how each 30 minute interval compares to the rolling average over the course of a day.

In this **specific** demo dataset, we have exactly one record per 30 minutes, so we could use a fixed window size of 12 (12 periods per market day). But in practice, we often have messy data with inconsistent time frequencies -- not to mention some missing or duplicated data. Fixed time windows can potentially corrupt our analysis.

Fortunately, solving this problem is actually pretty easy with pandas. We can use a fixed **time window** per ticker rather than fixed number of records.

Unfortunately, it's **pretty** slow.

In [ ]:
%time result = df.set_index("datetime").sort_index().groupby("ticker").rolling("1D").mean().reset_index()
result.head()

**About** a 20 to 30 seconds just for a single question. That's not ideal.  Especially for what comes next.

Now that we have an average price per day, let's do this again to get the Simple Moving Averages (SMA) usually used in stock analysis.  We'll first do the 50 Day SMA and then the 200 Day SMA.

In [ ]:
%time fiftyDay = df.set_index("datetime").sort_index().groupby("ticker").rolling("50D").mean().reset_index()
fiftyDay.head()

In [ ]:
%time twoHunDay = df.set_index("datetime").sort_index().groupby("ticker").rolling("200D").mean().reset_index()
twoHunDay.head()

This took just about a minute and a half to get to just about where you can plot.  

Perfect time to switch to cudf.pandas. Let's do that and then run the **same code**.

# Analysis with cuDF Pandas

Typically, you should load the `cudf.pandas` extension as the first step in your notebook, before importing any modules. Here, we explicitly restart the kernel to simulate that behavior.

# Timing with cudf.pandas

Since cudf.pandas was loaded at the top, all operations above were already GPU-accelerated.
The cells below re-run with explicit timing to show the speedup.

In [ ]:
import pandas as pd

We'll run the same code as above to get a feel what GPU-acceleration brings to pandas workflows.

In [ ]:
%time df = pd.read_parquet("usa_stocks_30m.parquet")
df.info()

In [ ]:
df.head()

Let's look at the more detailed summary statistics for the the numeric data like we did before.

In [ ]:
df = df.iloc[:18000000]
df.shape

In [ ]:
%time df.describe()

First things first, we can see that results are the same, even though we're now using the GPU. Great.

Second, that was much quicker, even with a little overhead from this being my first GPU-accelerated operation on the data.

Let's do the groupby aggregations.

In [ ]:
%time df.groupby("ticker").agg({"datetime": ["min", "max", "count"]})

In [ ]:
%%time
df[["year", "week", "day"]] = df.datetime.dt.isocalendar()
df.groupby(["ticker", "year", "week"]).agg({"close": ["min", "max"]})

20-50x faster with the same code. Nice!

Let's do the long-running groupby rolling operation.

In [ ]:
%time result = df.set_index("datetime").sort_index().groupby("ticker").rolling("1D").mean().reset_index()
result.head()

Nice! 25+ seconds down to about < 2 seconds.

Let's finish off those SMAs.  While we do that, let's profile the activity so you can see where the CPU and GPU

We've only used pandas so far, and things have worked smoothly.

But workflows often use multiple libraries, many of which are designed to accept pandas inputs. Fortunately, cudf.pandas works here, too.

In [ ]:
%time fiftyDay = df.set_index("datetime").sort_index().groupby("ticker").rolling("50D").mean().reset_index()
fiftyDay.head()

In [ ]:
%time twoHunDay = df.set_index("datetime").sort_index().groupby("ticker").rolling("200D").mean().reset_index()
twoHunDay.head()

Awesome, `cudf.pandas` got the same workflow done in about 10 seconds!

It is also important to understand that this speed difference increases as you add all the cells and increase complexity.  To do all a 200 Day SMA on all 36M cells will take **<12 seconds** on a the T4 using `cuDF.pandas` and well over a minute on CPU.

## Using third-party libraries with cudf.pandas


When using cudf.pandas, you can pass pandas objects to third-party libraries just like you would when using regular Pandas.

For example, we can pass this dataset to the visualization library plotnine and plot the yearly average closing price of the ticker `GOOG`.  Let's join the results of our groupbys above, form the data so that `plotnine` can ingest it, and chart this data out!

In [ ]:
result = result.join(fiftyDay, rsuffix='_50')
result = result.join(twoHunDay, rsuffix='_200')

In [ ]:
result.head()

In [ ]:
goog_closing_value = result.loc[result.ticker == "GOOG"]
goog_closing_value.head()

In [ ]:
goog_closing_value_p9 = goog_closing_value.melt(
    id_vars="datetime",
    value_vars=["close", "close_50", "close_200"],
    var_name="SMA",
    value_name="price"
).dropna()

goog_closing_value_p9.head()

Now let's bring in `plotnine` and visually analyze the stock and it's performance versus the 50 and 200 day moving averages!

In [ ]:
from plotnine import *

In [ ]:
# Gallery, lines
(
    ggplot(goog_closing_value_p9, aes(x="datetime", y="price", color="SMA"))
    + geom_line()
    # Styling
    + scale_x_datetime(date_breaks="1 year", date_labels="%Y")
    + theme_538()
)

# Summary

With `cudf.pandas`, you can keep using pandas as your primary dataframe library if you enjoy using it but want faster performance. When things start to get a little slow, just load the `cudf.pandas` and run your existing code on a GPU!

If you like Google Colab and want to get peak cudf.pandas performance to process even larger datasets, Google Colab's paid tier includes both L4 and A100 GPUs (in addition to the T4 GPU this demo notebook is using).

To learn more about cudf.pandas, we encourage you to visit [rapids.ai/cudf-pandas](https://rapids.ai/cudf-pandas).